In [0]:
from config.f1_config import add_ingestion_metadata,CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType,TimestampType, FloatType
import pyspark.sql.functions as F

SILVER_TABLE = "f1_workspace.silver.results"

In [0]:
df_bronze_results = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.results")
display(df_bronze_results)



TRANSFORM DATA


In [0]:
def transform_results_silver(df_bronze):

    return (
        df_bronze.select(
            F.trim(F.col("Driver.driverId")).alias("Driver_Id"),
            F.trim(F.col("Constructor.constructorId")).alias("Constructor_Id"),
            F.trim(F.col("season")).cast("int").alias("Season"),
            F.trim(F.col("round")).cast("int").alias("Round"),
            F.trim(F.initcap(F.col("raceName"))).alias("Race_Name"),
            F.trim(F.col("grid")).cast("int").alias("Grid"),
            F.trim(F.col("position")).cast("int").alias("Position"),
            F.trim(F.col("positionText")).alias("Position_Text"),
            F.trim(F.col("points")).cast("double").alias("Points"),
            F.trim(F.col("laps")).cast("int").alias("Laps"),
            F.trim(F.col("status")).alias("Status"),
            F.trim(F.col("FastestLap.rank")).cast("int").alias("Fastest_Lap_Rank"),
        )
        .withColumn("silver_processed_timestamp", F.current_timestamp())
    )


df_silver_results = transform_results_silver(df_bronze_results)
display(df_silver_results)


In [0]:
df_silver_results.write.format("delta")\
    .mode("append")\
    .option("overwriteSchema", True)\
    .saveAsTable(SILVER_TABLE)